In [5]:
import pandas as pd
import numpy as np

In [64]:
paths = ({
    'invoice_level_table': '../data/D3_invoice-level table.csv',
    'customer_features': '../data/D6_customer_features.csv'
})

In [3]:
df = pd.read_csv(paths['invoice_level_table'])

In [9]:
# generate toy dataset for merging
df1 = pd.DataFrame({
    'key': np.random.choice(['A', 'B', 'C'], size=10),
    'value': np.random.randint(1, 100, size=10)
})
df2 = pd.DataFrame({
    'key': np.random.choice(['A', 'B', 'D'], size=10),
    'value': np.random.randint(1, 100, size=10)
})

#### 1. `merge` two datasets

In [17]:
# perform different types of merges
inner = pd.merge(df1, df2, on='key', how='inner')
left = pd.merge(df1, df2, on='key', how='left')
right = pd.merge(df1, df2, on='key', how='right')
outer = pd.merge(df1, df2, on='key', how='outer')

# compare row counts
print(f"Inner join row count: {len(inner)}")
print(f"Left join row count: {len(left)}")
print(f"Right join row count: {len(right)}")
print(f"Outer join row count: {len(outer)}")
print('This is because, inner join only keeps rows with matching keys in both tables, \nwhile left join keeps all rows from the left table and fills missing values for non-matching keys, \nright join keeps all rows from the right table and fills missing values for non-matching keys, \nand outer join keeps all rows from both tables and fills missing values for non-matching keys.')
print(f'Note that {len(inner)} = len(left) - {len(left) - len(inner)} = len(right) - {len(right) - len(inner)} = {len(outer)} - {len(outer) - len(inner)}')

Inner join row count: 20
Left join row count: 25
Right join row count: 23
Outer join row count: 28
This is because, inner join only keeps rows with matching keys in both tables, 
while left join keeps all rows from the left table and fills missing values for non-matching keys, 
right join keeps all rows from the right table and fills missing values for non-matching keys, 
and outer join keeps all rows from both tables and fills missing values for non-matching keys.
Note that 20 = len(left) - 5 = len(right) - 3 = 28 - 8


In [24]:
# indicator shows the source of each row
ind_true = pd.merge(df1, df2, on='key', how='left', indicator=True)
ind_true.head()

,key,value_x,value_y,_merge
0,A,92,70.0,both
1,C,43,NaN,left_only
2,C,78,NaN,left_only
3,B,98,79.0,both
4,B,98,19.0,both


In [30]:
# validate='m:1' ensures that each key in the right table (df2) matches at most one key in the left table (df1)
# will return MergeError if not satisfied
pd.merge(df1, df2, on='key', how='right', validate='m:1')

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 key
  B
  B
  D
  B
  B ...

#### 2. merge vs. join vs. concat
merge() is a module function, not belong to either dataset
.join() is an instance method, belong to the left dataset
concat() stitch two dataframes together, no shared parameter

## Lab: Build customer features

#### 1. customer features table

In [ ]:
# total revenue by customer
df_rev = df.groupby('customer_id')['invoice_revenue'].sum().reset_index()
# number of invoices
df_inv = df.groupby('customer_id')['n_lines'].sum().reset_index()
# average AOV (average order value)
df_aov = df.groupby('customer_id')['invoice_revenue'].mean().reset_index()
# last purchase date
df_last = df.groupby('customer_id')['invoice_first_date'].max().reset_index()

# merge all features into one table, validate for duplicates
df_customer = pd.merge(df_rev, df_inv, on='customer_id', validate='1:1')
df_customer = pd.merge(df_customer, df_aov, on='customer_id', suffixes=('', '_aov'), validate='1:1')
df_customer = pd.merge(df_customer, df_last, on='customer_id', validate='1:1')

# rename columns for clarity
df_customer.rename(columns={
    'invoice_revenue': 'total_revenue',
    'n_lines': 'n_invoices',
    'invoice_revenue_aov': 'avg_aov',
    'invoice_first_date': 'last_purchase_date'
}, inplace=True)
df_customer.head()

,customer_id,total_revenue,n_invoices,avg_aov,last_purchase_date
0,12346.0,77183.60,1,77183.600000,2011-01-18 10:01:00
1,12347.0,4310.00,182,615.714286,2011-12-07 15:52:00
2,12348.0,1797.24,31,449.310000,2011-09-25 13:13:00
3,12349.0,1757.55,73,1757.550000,2011-11-21 09:51:00
4,12350.0,334.40,17,334.400000,2011-02-02 16:01:00


In [65]:
df_customer.to_csv(paths['customer_features'], index=False)

#### 2. anti-explosion diagnostics

In [55]:
# dupicate rate check
print(f"Duplicate customer_id in revenue table: {df_customer['customer_id'].duplicated().sum()}")
print(f"Duplicate customer_id in invoice table: {df['customer_id'].duplicated().sum()}")

Duplicate customer_id in revenue table: 0
Duplicate customer_id in invoice table: 14194


#### 3. merge `df_customer` with invoice table

In [63]:
pd.merge(df, df_customer, on='customer_id', how='left', validate='m:1')

,InvoiceNo,invoice_first_date,customer_id,country,invoice_revenue,n_lines,total_revenue,n_invoices,avg_aov,last_purchase_date
0,536365,2010-12-01 08:26:00,17850.0,United Kingdom,139.12,7,5391.21,297,158.565000,2010-12-02 15:27:00
1,536366,2010-12-01 08:28:00,17850.0,United Kingdom,22.20,2,5391.21,297,158.565000,2010-12-02 15:27:00
2,536367,2010-12-01 08:34:00,13047.0,United Kingdom,278.73,12,3237.54,172,323.754000,2011-11-08 12:06:00
3,536368,2010-12-01 08:34:00,13047.0,United Kingdom,70.05,4,3237.54,172,323.754000,2011-11-08 12:06:00
4,536369,2010-12-01 08:35:00,13047.0,United Kingdom,17.85,1,3237.54,172,323.754000,2011-11-08 12:06:00
...,...,...,...,...,...,...,...,...,...,...
18527,581583,2011-12-09 12:23:00,13777.0,United Kingdom,124.60,2,25977.16,197,787.186667,2011-12-09 12:25:00
18528,581584,2011-12-09 12:25:00,13777.0,United Kingdom,140.64,2,25977.16,197,787.186667,2011-12-09 12:25:00
18529,581585,2011-12-09 12:31:00,15804.0,United Kingdom,329.05,21,4206.39,262,323.568462,2011-12-09 12:31:00
18530,581586,2011-12-09 12:49:00,13113.0,United Kingdom,339.20,4,12245.96,200,510.248333,2011-12-09 12:49:00
